In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [2]:
cancer_data = load_breast_cancer()
data_frame = pd.DataFrame(cancer_data.data, columns=cancer_data.feature_names)
data_frame['target'] = cancer_data.target
display(data_frame.head())
print('Dataset shape:', data_frame.shape)
print('Class distribution:')
print(data_frame['target'].value_counts().sort_index())

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


Dataset shape: (569, 31)
Class distribution:
target
0    212
1    357
Name: count, dtype: int64


In [3]:
X = data_frame.drop('target', axis=1)
y = data_frame['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Training size:', X_train.shape[0])
print('Testing size:', X_test.shape[0])

Training size: 455
Testing size: 114


In [4]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [5]:
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X_train_scaled, y_train)
logistic_predictions = logistic_model.predict(X_test_scaled)
logistic_accuracy = accuracy_score(y_test, logistic_predictions)
print('Logistic Regression accuracy:', logistic_accuracy)

Logistic Regression accuracy: 0.9824561403508771


In [6]:
tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_model.fit(X_train, y_train)
tree_predictions = tree_model.predict(X_test)
tree_accuracy = accuracy_score(y_test, tree_predictions)
print('Decision Tree accuracy:', tree_accuracy)

Decision Tree accuracy: 0.9385964912280702


In [7]:
accuracy_comparison = pd.DataFrame({'Model': ['Logistic Regression', 'Decision Tree'], 'Accuracy': [logistic_accuracy, tree_accuracy]})
accuracy_comparison['Accuracy Percentage'] = accuracy_comparison['Accuracy'] * 100
display(accuracy_comparison)
higher_accuracy_model = accuracy_comparison.loc[accuracy_comparison['Accuracy'].idxmax(), 'Model']
print('Model with higher accuracy:', higher_accuracy_model)
accuracy_comparison.to_csv('accuracy_comparison.csv', index=False)

,Model,Accuracy,Accuracy Percentage
0,Logistic Regression,0.982456,98.245614
1,Decision Tree,0.938596,93.859649


Model with higher accuracy: Logistic Regression


In [8]:
logistic_precision = precision_score(y_test, logistic_predictions)
logistic_recall = recall_score(y_test, logistic_predictions)
logistic_f1 = f1_score(y_test, logistic_predictions)
tree_precision = precision_score(y_test, tree_predictions)
tree_recall = recall_score(y_test, tree_predictions)
tree_f1 = f1_score(y_test, tree_predictions)
metrics_comparison = pd.DataFrame({'Model': ['Logistic Regression', 'Decision Tree'], 'Precision': [logistic_precision, tree_precision], 'Recall': [logistic_recall, tree_recall], 'F1-score': [logistic_f1, tree_f1]})
display(metrics_comparison)
print('Positive class: 1 (benign)')
print('Logistic Regression confusion matrix:')
print(confusion_matrix(y_test, logistic_predictions))
print('Decision Tree confusion matrix:')
print(confusion_matrix(y_test, tree_predictions))

,Model,Precision,Recall,F1-score
0,Logistic Regression,0.986111,0.986111,0.986111
1,Decision Tree,0.957746,0.944444,0.951049


Positive class: 1 (benign)
Logistic Regression confusion matrix:
[[41  1]
 [ 1 71]]
Decision Tree confusion matrix:
[[39  3]
 [ 4 68]]


In [9]:
logistic_train_accuracy = accuracy_score(y_train, logistic_model.predict(X_train_scaled))
tree_train_accuracy = accuracy_score(y_train, tree_model.predict(X_train))
generalization_comparison = pd.DataFrame({'Model': ['Logistic Regression', 'Decision Tree'], 'Training Accuracy': [logistic_train_accuracy, tree_train_accuracy], 'Testing Accuracy': [logistic_accuracy, tree_accuracy]})
generalization_comparison['Train-Test Gap'] = generalization_comparison['Training Accuracy'] - generalization_comparison['Testing Accuracy']
display(generalization_comparison)
for _, row in generalization_comparison.iterrows():
    if row['Train-Test Gap'] > 0.05:
        print(row['Model'], 'has a noticeable train-test gap that may indicate overfitting.')
    else:
        print(row['Model'], 'has a small train-test gap, but one split alone cannot confirm generalization.')

,Model,Training Accuracy,Testing Accuracy,Train-Test Gap
0,Logistic Regression,0.989011,0.982456,0.006555
1,Decision Tree,0.986813,0.938596,0.048217


Logistic Regression has a small train-test gap, but one split alone cannot confirm generalization.
Decision Tree has a small train-test gap, but one split alone cannot confirm generalization.


In [10]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
cross_validation = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
logistic_pipeline = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000, random_state=42))])
tree_cross_validation_model = DecisionTreeClassifier(max_depth=4, random_state=42)
logistic_cross_validation_scores = cross_val_score(logistic_pipeline, X, y, cv=cross_validation, scoring='accuracy')
tree_cross_validation_scores = cross_val_score(tree_cross_validation_model, X, y, cv=cross_validation, scoring='accuracy')
cross_validation_comparison = pd.DataFrame({'Model': ['Logistic Regression', 'Decision Tree'], 'Fold 1 Accuracy': [logistic_cross_validation_scores[0], tree_cross_validation_scores[0]], 'Fold 2 Accuracy': [logistic_cross_validation_scores[1], tree_cross_validation_scores[1]], 'Fold 3 Accuracy': [logistic_cross_validation_scores[2], tree_cross_validation_scores[2]], 'Fold 4 Accuracy': [logistic_cross_validation_scores[3], tree_cross_validation_scores[3]], 'Fold 5 Accuracy': [logistic_cross_validation_scores[4], tree_cross_validation_scores[4]], 'Mean Accuracy': [logistic_cross_validation_scores.mean(), tree_cross_validation_scores.mean()], 'Standard Deviation': [logistic_cross_validation_scores.std(), tree_cross_validation_scores.std()]})
display(cross_validation_comparison)
print('Logistic Regression fold accuracies:', np.round(logistic_cross_validation_scores, 4))
print(f'Logistic Regression mean accuracy: {logistic_cross_validation_scores.mean():.4f}')
print(f'Logistic Regression standard deviation: {logistic_cross_validation_scores.std():.4f}')
print('Decision Tree fold accuracies:', np.round(tree_cross_validation_scores, 4))
print(f'Decision Tree mean accuracy: {tree_cross_validation_scores.mean():.4f}')
print(f'Decision Tree standard deviation: {tree_cross_validation_scores.std():.4f}')

,Model,Fold 1 Accuracy,Fold 2 Accuracy,Fold 3 Accuracy,Fold 4 Accuracy,Fold 5 Accuracy,Mean Accuracy,Standard Deviation
0,Logistic Regression,0.973684,0.947368,0.964912,0.991228,0.991150,0.973669,0.016627
1,Decision Tree,0.956140,0.868421,0.921053,0.929825,0.938053,0.922698,0.029508


Logistic Regression fold accuracies: [0.9737 0.9474 0.9649 0.9912 0.9912]
Logistic Regression mean accuracy: 0.9737
Logistic Regression standard deviation: 0.0166
Decision Tree fold accuracies: [0.9561 0.8684 0.9211 0.9298 0.9381]
Decision Tree mean accuracy: 0.9227
Decision Tree standard deviation: 0.0295


In [11]:
logistic_false_predictions = int((y_test != logistic_predictions).sum())
tree_false_predictions = int((y_test != tree_predictions).sum())
print('1.', higher_accuracy_model, 'had the higher testing accuracy in this split.')
print(f'2. Logistic Regression made {logistic_false_predictions} false predictions, while Decision Tree made {tree_false_predictions}.')
print(f'3. Logistic Regression had a train-test accuracy gap of {logistic_train_accuracy - logistic_accuracy:.4f} and Decision Tree had a gap of {tree_train_accuracy - tree_accuracy:.4f}.')
print(f'4. In 5-fold cross-validation, Logistic Regression averaged {logistic_cross_validation_scores.mean():.4f} with a standard deviation of {logistic_cross_validation_scores.std():.4f}, while Decision Tree averaged {tree_cross_validation_scores.mean():.4f} with a standard deviation of {tree_cross_validation_scores.std():.4f}.')
print('5. Cross-validation adds evidence beyond one split, but these results do not make either model universally better.')

1. Logistic Regression had the higher testing accuracy in this split.
2. Logistic Regression made 2 false predictions, while Decision Tree made 7.
3. Logistic Regression had a train-test accuracy gap of 0.0066 and Decision Tree had a gap of 0.0482.
4. In 5-fold cross-validation, Logistic Regression averaged 0.9737 with a standard deviation of 0.0166, while Decision Tree averaged 0.9227 with a standard deviation of 0.0295.
5. Cross-validation adds evidence beyond one split, but these results do not make either model universally better.
